In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import shutil
import glob

# Initialize Spark
spark = SparkSession.builder \
    .appName("Year_Based_Incremental_Splitting") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# ============================================================================
# SECTION 1: LOAD ORIGINAL DATA
# ============================================================================
print("\n" + "="*80)
print("📂 LOADING ORIGINAL TPC-H DATA")
print("="*80)

try:
    orders = spark.read.parquet("../../../data/raw/tables/original/orders.parquet")
    lineitem = spark.read.parquet("../../../data/raw/tables/original/lineitem.parquet")
    print("✅ Data loaded successfully.")
except Exception as e:
    print(f"❌ Error loading files: {e}")
    raise

# ============================================================================
# SECTION 2: SPLIT DATA BY YEAR (HISTORICAL YEARS / INCREMENTAL YEARS)
# ============================================================================
print("\n" + "="*80)
print("✂️  SPLITTING DATA BY YEAR (HISTORICAL VS INCREMENTAL)")
print("="*80)

# Extract year from o_orderdate
orders_with_year = orders.withColumn("year", F.year("o_orderdate"))

# Get distinct years sorted
distinct_years = orders_with_year.select("year").distinct().orderBy("year").collect()
all_years = [row[0] for row in distinct_years]

print(f"All years in data: {all_years}")

# Define split strategy - let's make first 60% of YEARS as historical, last 40% as incremental
# (or you can specify specific years - uncomment alternative approach below)

# Option 1: Split by percentage of years
num_years = len(all_years)
historical_year_count = int(num_years * 0.6)
historical_years = all_years[:historical_year_count]
incremental_years = all_years[historical_year_count:]

# Option 2: Alternative - specify explicit year ranges (uncomment to use)
# historical_years = [1992, 1993, 1994, 1995]  # First 4 years
# incremental_years = [1996, 1997, 1998]        # Last 3 years

print(f"\nSplit strategy: {historical_year_count} historical years, {num_years - historical_year_count} incremental years")
print(f"Historical years (base layer): {historical_years}")
print(f"Incremental years (batches): {incremental_years}")

# Split orders by year
historical_orders = orders_with_year.filter(F.col("year").isin(historical_years)).drop("year").repartition(5)
recent_orders = orders_with_year.filter(F.col("year").isin(incremental_years)).drop("year").repartition(5)

# Get counts
historical_orders_count = historical_orders.count()
recent_orders_count = recent_orders.count()
total_orders_count = historical_orders_count + recent_orders_count

print(f"\nOrders distribution:")
print(f"  Historical orders: {historical_orders_count:,} ({historical_orders_count/total_orders_count*100:.1f}%)")
print(f"  Incremental orders: {recent_orders_count:,} ({recent_orders_count/total_orders_count*100:.1f}%)")

# Create temporary views for joins
historical_orders.createOrReplaceTempView("temp_orders_hist")
recent_orders.createOrReplaceTempView("temp_orders_recent")
lineitem.createOrReplaceTempView("temp_lineitem_all")

# Distribute lineitems based on their parent orders' years
historical_lineitem = spark.sql("""
    SELECT l.* FROM temp_lineitem_all l
    INNER JOIN temp_orders_hist o ON l.l_orderkey = o.o_orderkey
""").repartition(5)

recent_lineitem = spark.sql("""
    SELECT l.* FROM temp_lineitem_all l
    INNER JOIN temp_orders_recent o ON l.l_orderkey = o.o_orderkey
""").repartition(5)

# Verify the split
hist_lineitem_count = historical_lineitem.count()
recent_lineitem_count = recent_lineitem.count()
total_lineitems = hist_lineitem_count + recent_lineitem_count
print(f"\nLineitem distribution:")
print(f"  Historical lineitems: {hist_lineitem_count:,} ({hist_lineitem_count/total_lineitems*100:.1f}%)")
print(f"  Incremental lineitems: {recent_lineitem_count:,} ({recent_lineitem_count/total_lineitems*100:.1f}%)")

# ============================================================================
# SECTION 3: FUNCTION TO WRITE SINGLE PARQUET FILES
# ============================================================================
def write_single_parquet(df, output_path):
    """Write DataFrame as a single parquet file (not a directory)"""
    # Create temporary directory
    temp_dir = output_path + "_temp"
    
    # Write as single parquet file (Spark always creates a directory)
    df.coalesce(1).write.mode("overwrite").parquet(temp_dir)
    
    # Find the actual parquet file in the temp directory
    for file in os.listdir(temp_dir):
        if file.endswith('.parquet'):
            # Move the parquet file to the desired location
            shutil.move(os.path.join(temp_dir, file), output_path)
            break
    
    # Remove the temporary directory
    shutil.rmtree(temp_dir)
    
    return output_path

# ============================================================================
# SECTION 4: SAVE TO WORK_DATA AS SINGLE FILES
# ============================================================================
print("\n" + "="*80)
print("💾 SAVING DATA TO WORK_DATA AS SINGLE FILES")
print("="*80)

# Define output paths - NOW DIRECTLY TO WORK_DATA
work_data_path = "../../../data/raw/tables/work_data"
base_output_path = os.path.join(work_data_path, "bases")
increments_output_path = os.path.join(work_data_path, "increments")

# Clean existing work_data if it exists
if os.path.exists(work_data_path):
    print(f"\n⚠️ Warning: {work_data_path} already exists.")
    response = input("Do you want to overwrite it? (yes/no): ")
    if response.lower() != 'yes':
        print("❌ Operation cancelled.")
        spark.stop()
        exit()
    shutil.rmtree(work_data_path)

# Create fresh directories
os.makedirs(base_output_path, exist_ok=True)
os.makedirs(increments_output_path, exist_ok=True)

print(f"\n📁 Created structure:")
print(f"   {base_output_path}")
print(f"   {increments_output_path}")

# Save historical data as single files (base layer)
print("\n📦 Saving historical base layer (all historical years) as single files...")

# Historical orders
historical_orders_output = os.path.join(base_output_path, "orders_base.parquet")
write_single_parquet(historical_orders, historical_orders_output)
print(f"   ✅ Saved: orders_base.parquet ({historical_orders_count:,} records)")

# Historical lineitems
historical_lineitem_output = os.path.join(base_output_path, "lineitem_base.parquet")
write_single_parquet(historical_lineitem, historical_lineitem_output)
print(f"   ✅ Saved: lineitem_base.parquet ({hist_lineitem_count:,} records)")

print(f"  ✅ Historical data saved to: {base_output_path}")

# ============================================================================
# SECTION 5: CREATE INCREMENTAL BATCHES - ONE BATCH PER YEAR
# ============================================================================
print("\n" + "="*80)
print("📊 CREATING INCREMENTAL BATCHES (ONE BATCH PER YEAR)")
print("="*80)

# Add year column to recent_orders for easier filtering
recent_orders_with_year = recent_orders.withColumn("year", F.year("o_orderdate"))

print(f"\nCreating separate batches for each incremental year: {incremental_years}")

# Track batch statistics
batch_stats = []

# Create one batch per incremental year
for year in incremental_years:
    print(f"\n📁 Batch for Year {year}:")
    
    # Get orders for this year
    batch_orders = recent_orders_with_year.filter(F.col("year") == year).drop("year").repartition(5)
    batch_orders_count = batch_orders.count()
    print(f"    Orders: {batch_orders_count:,}")
    
    # Get corresponding lineitems
    batch_lineitem = recent_lineitem.join(
        batch_orders.select("o_orderkey"), 
        recent_lineitem.l_orderkey == batch_orders.o_orderkey, 
        "inner"
    ).select(recent_lineitem["*"]).repartition(5)
    
    batch_lineitem_count = batch_lineitem.count()
    print(f"    Lineitems: {batch_lineitem_count:,}")
    
    # Create batch folder for this year
    batch_folder = os.path.join(increments_output_path, f"batch_{year}")
    os.makedirs(batch_folder, exist_ok=True)
    
    # Save as single parquet files
    orders_output = os.path.join(batch_folder, "orders.parquet")
    lineitem_output = os.path.join(batch_folder, "lineitem.parquet")
    
    write_single_parquet(batch_orders, orders_output)
    write_single_parquet(batch_lineitem, lineitem_output)
    
    # Get file sizes
    orders_size_mb = os.path.getsize(orders_output) / (1024 * 1024)
    lineitem_size_mb = os.path.getsize(lineitem_output) / (1024 * 1024)
    
    print(f"    ✅ Saved: orders.parquet ({orders_size_mb:.2f} MB)")
    print(f"    ✅ Saved: lineitem.parquet ({lineitem_size_mb:.2f} MB)")
    
    batch_stats.append({
        "year": year,
        "orders": batch_orders_count,
        "lineitems": batch_lineitem_count,
        "orders_size_mb": orders_size_mb,
        "lineitem_size_mb": lineitem_size_mb
    })
    
    print(f"    ✅ Batch for year {year} complete")

# ============================================================================
# SECTION 6: ADD ALL ORIGINAL TPC-H TABLES TO WORK_DATA
# ============================================================================
print("\n" + "="*80)
print("📦 ADDING ALL ORIGINAL TPC-H TABLES TO WORK_DATA")
print("="*80)

original_path = "/home/jovyan/data/raw/tables/original"
original_data_folder = os.path.join(work_data_path, "original_data")

if os.path.exists(original_path):
    # Create original_data folder
    os.makedirs(original_data_folder, exist_ok=True)
    
    print(f"\n📁 Copying all original TPC-H tables from {original_path} to {original_data_folder}/")
    print("-" * 60)
    
    # Define all TPC-H tables to copy
    tpch_tables = [
        'customer.parquet',
        'lineitem.parquet',
        'nation.parquet',
        'orders.parquet',
        'part.parquet',
        'partsupp.parquet',
        'region.parquet',
        'supplier.parquet'
    ]
    
    copied_count = 0
    total_size = 0
    missing_tables = []
    
    for table in tpch_tables:
        source_path = os.path.join(original_path, table)
        dest_path = os.path.join(original_data_folder, table)
        
        if os.path.exists(source_path):
            try:
                # Get file size
                file_size = os.path.getsize(source_path)
                total_size += file_size
                file_size_mb = file_size / (1024 * 1024)
                
                print(f"   📄 Copying: {table} ({file_size_mb:.2f} MB)")
                
                # Copy the file
                shutil.copy2(source_path, dest_path)
                
                # Verify copy was successful
                if os.path.exists(dest_path):
                    print(f"      ✅ Successfully copied")
                    copied_count += 1
                else:
                    print(f"      ❌ Failed to copy")
                    missing_tables.append(table)
                    
            except Exception as e:
                print(f"      ❌ Error copying {table}: {e}")
                missing_tables.append(table)
        else:
            print(f"   ⚠️  Missing: {table} (not found in original folder)")
            missing_tables.append(table)
    
    # Calculate total size in GB
    total_size_gb = total_size / (1024 * 1024 * 1024)
    
    print("\n" + "="*80)
    print("📊 COPY SUMMARY")
    print("="*80)
    print(f"✅ Successfully copied: {copied_count}/{len(tpch_tables)} tables")
    print(f"   Total size: {total_size_gb:.2f} GB")
    
    if missing_tables:
        print(f"\n⚠️  Missing or failed tables ({len(missing_tables)}):")
        for table in missing_tables:
            print(f"   - {table}")
    
    # Verify copied files with Spark
    print("\n" + "="*80)
    print("🔍 VERIFYING COPIED DATA WITH SPARK")
    print("="*80)
    
    verification_results = []
    
    for table in tpch_tables:
        file_path = os.path.join(original_data_folder, table)
        if os.path.exists(file_path):
            try:
                # Read the parquet file
                df = spark.read.parquet(file_path)
                
                # Get basic statistics
                record_count = df.count()
                column_count = len(df.columns)
                
                print(f"\n   📊 {table}:")
                print(f"      ✅ File successfully read")
                print(f"      📊 Records: {record_count:,}")
                print(f"      📋 Columns: {column_count}")
                
                verification_results.append({
                    "table": table.replace('.parquet', ''),
                    "status": "OK",
                    "records": record_count,
                    "columns": column_count
                })
                
            except Exception as e:
                print(f"\n   ❌ {table}: Error reading - {e}")
                verification_results.append({
                    "table": table.replace('.parquet', ''),
                    "status": "ERROR",
                    "error": str(e)
                })
    
    # Display verification summary
    if verification_results:
        print("\n" + "="*80)
        print("📋 VERIFICATION SUMMARY")
        print("="*80)
        print(f"\n{'Table':<15} {'Status':<8} {'Records':<15} {'Columns':<10}")
        print("-" * 55)
        
        for result in verification_results:
            if result['status'] == 'OK':
                print(f"{result['table']:<15} ✅       {result['records']:<15,} {result['columns']:<10}")
            else:
                print(f"{result['table']:<15} ❌       {'N/A':<15} {'N/A':<10}")
    
else:
    print(f"❌ Error: Original path not found at {original_path}")
    print("   Cannot copy original tables to work_data")

# ============================================================================
# SECTION 7: VERIFY DATA INTEGRITY
# ============================================================================
print("\n" + "="*80)
print("🔍 VERIFYING DATA INTEGRITY")
print("="*80)

# Verify historical data
historical_orders_count = spark.read.parquet(historical_orders_output).count()
historical_lineitem_count = spark.read.parquet(historical_lineitem_output).count()

# Verify recent data
recent_orders_total = sum(batch['orders'] for batch in batch_stats)
recent_lineitem_total = sum(batch['lineitems'] for batch in batch_stats)

print(f"\nOrders:")
print(f"  Historical (years {historical_years}): {historical_orders_count:,}")
print(f"  Incremental (years {incremental_years}): {recent_orders_total:,}")
print(f"  Total: {historical_orders_count + recent_orders_total:,}")
print(f"  Original total: {total_orders_count:,}")
print(f"  ✅ Match: {historical_orders_count + recent_orders_total == total_orders_count}")

print(f"\nLineitems:")
print(f"  Historical: {historical_lineitem_count:,}")
print(f"  Incremental: {recent_lineitem_total:,}")
print(f"  Total: {historical_lineitem_count + recent_lineitem_total:,}")
print(f"  Original total: {total_lineitems:,}")
print(f"  ✅ Match: {historical_lineitem_count + recent_lineitem_total == total_lineitems}")

# Verify year distribution
print(f"\nYear Distribution Check:")
for year in incremental_years:
    year_orders = spark.read.parquet(os.path.join(increments_output_path, f"batch_{year}", "orders.parquet")).count()
    expected_orders = next((batch['orders'] for batch in batch_stats if batch['year'] == year), 0)
    print(f"  Year {year}: {year_orders:,} orders (expected: {expected_orders:,}) ✅" if year_orders == expected_orders else f"  Year {year}: {year_orders:,} orders (expected: {expected_orders:,}) ❌")

# ============================================================================
# SECTION 8: CLEANUP - REMOVE INCREMENTAL_INGESTION_COMBINED (if exists)
# ============================================================================
print("\n" + "="*80)
print("🧹 CLEANUP: Removing Combined Incremental Ingestion Data")
print("="*80)

combined_path = "../../../data/raw/tables/incremental_ingestion_combined"

if os.path.exists(combined_path):
    # Calculate size before deletion
    combined_size = 0
    combined_files = 0
    for root, dirs, files in os.walk(combined_path):
        for file in files:
            combined_size += os.path.getsize(os.path.join(root, file))
            combined_files += 1
    
    combined_size_gb = combined_size / (1024 * 1024 * 1024)
    print(f"\n📊 Combined folder size: {combined_size_gb:.2f} GB ({combined_files:,} files)")
    
    # Delete the folder
    print(f"\n🗑️  Deleting: {combined_path}")
    shutil.rmtree(combined_path)
    print("   ✅ Combined folder deleted successfully!")
    
    # Clean any remaining temp files
    temp_patterns = [
        "../../../data/raw/tables/temp_*",
        "../../../data/raw/tables/*/temp_*"
    ]
    
    temp_removed = 0
    for temp_pattern in temp_patterns:
        for temp_file in glob.glob(temp_pattern):
            try:
                if os.path.isfile(temp_file):
                    os.remove(temp_file)
                elif os.path.isdir(temp_file):
                    shutil.rmtree(temp_file)
                temp_removed += 1
            except:
                pass
    
    if temp_removed > 0:
        print(f"   🗑️  Removed {temp_removed} temporary files/directories")
else:
    print(f"📁 Combined folder not found (already cleaned)")

# ============================================================================
# SECTION 9: FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✨ YEAR-BASED DATA SPLITTER PIPELINE COMPLETE!")
print("="*80)

# Display final work_data structure
print("\n📁 FINAL WORK_DATA STRUCTURE:")
print("="*80)

def print_structure(path, prefix="", max_depth=2, current_depth=0):
    """Print directory structure"""
    if current_depth > max_depth or not os.path.exists(path):
        return
    
    items = sorted([i for i in os.listdir(path) if not i.startswith('.')])
    for i, item in enumerate(items):
        item_path = os.path.join(path, item)
        is_last = i == len(items) - 1
        
        if os.path.isfile(item_path):
            if item.endswith('.parquet'):
                size_mb = os.path.getsize(item_path) / (1024 * 1024)
                print(f"{prefix}{'└── ' if is_last else '├── '}{item} ({size_mb:.2f} MB)")
        else:
            # Calculate folder size
            folder_size = 0
            for root, dirs, files in os.walk(item_path):
                for file in files:
                    folder_size += os.path.getsize(os.path.join(root, file))
            folder_size_mb = folder_size / (1024 * 1024)
            folder_size_gb = folder_size_mb / 1024
            
            if folder_size_gb >= 1:
                size_str = f"{folder_size_gb:.2f} GB"
            else:
                size_str = f"{folder_size_mb:.2f} MB"
            
            print(f"{prefix}{'└── ' if is_last else '├── '}{item}/ - {size_str}")
            print_structure(item_path, prefix + ("    " if is_last else "│   "), max_depth, current_depth + 1)

print_structure(work_data_path, max_depth=3)

print("\n" + "="*80)
print("📊 PIPELINE SUMMARY")
print("="*80)
print(f"✅ Historical data (years {historical_years}): {base_output_path}")
print(f"   - orders_base.parquet: {historical_orders_count:,} records")
print(f"   - lineitem_base.parquet: {historical_lineitem_count:,} records")
print(f"\n✅ Incremental batches (one per year): {increments_output_path}")
for batch in batch_stats:
    print(f"   - Year {batch['year']}: {batch['orders']:,} orders, {batch['lineitems']:,} lineitems ({batch['orders_size_mb']:.1f} MB / {batch['lineitem_size_mb']:.1f} MB)")
print(f"\n✅ Original data backup: {original_data_folder}")
print(f"   - {copied_count} tables copied")
print(f"\n✅ Combined folder cleaned: {not os.path.exists(combined_path)}")
print("\n🎯 Data is now split by YEAR (historical base + yearly increments)!")
print("   ✨ Historical years serve as the base layer")
print("   ✨ Each incremental year is its own batch")
print("   ✨ All files are single parquet files (not multi-part directories)")
print("="*80)

# Stop Spark session
spark.stop()
print("\n✅ Spark session stopped. Pipeline completed successfully!")


📂 LOADING ORIGINAL TPC-H DATA
✅ Data loaded successfully.

✂️  SPLITTING DATA BY YEAR (HISTORICAL VS INCREMENTAL)
All years in data: [1992, 1993, 1994, 1995, 1996, 1997, 1998]

Split strategy: 4 historical years, 3 incremental years
Historical years (base layer): [1992, 1993, 1994, 1995]
Incremental years (batches): [1996, 1997, 1998]

Orders distribution:
  Historical orders: 909,968 (60.7%)
  Incremental orders: 590,032 (39.3%)

Lineitem distribution:
  Historical lineitems: 3,640,678 (60.7%)
  Incremental lineitems: 2,360,537 (39.3%)

💾 SAVING DATA TO WORK_DATA AS SINGLE FILES

📁 Created structure:
   ../../../data/raw/tables/work_data/bases
   ../../../data/raw/tables/work_data/increments

📦 Saving historical base layer (all historical years) as single files...
   ✅ Saved: orders_base.parquet (909,968 records)
   ✅ Saved: lineitem_base.parquet (3,640,678 records)
  ✅ Historical data saved to: ../../../data/raw/tables/work_data/bases

📊 CREATING INCREMENTAL BATCHES (ONE BATCH PER Y

In [5]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.appName("Quick_Check").getOrCreate()

work_data_path = "../../../data/raw/tables/work_data"

# Check base layer
print("\n" + "="*60)
print("BASE LAYER (Historical Data)")
print("="*60)
base_orders = spark.read.parquet(os.path.join(work_data_path, "bases", "orders_base.parquet"))
base_max_orderkey = base_orders.agg(F.max("o_orderkey")).collect()[0][0]
base_max_orderdate = base_orders.agg(F.max("o_orderdate")).collect()[0][0]
print(f"Max o_orderkey: {base_max_orderkey}")
print(f"Max o_orderdate: {base_max_orderdate}")

# Check each increment (year batch)
print("\n" + "="*60)
print("INCREMENTAL BATCHES (By Year)")
print("="*60)

increments_path = os.path.join(work_data_path, "increments")
for batch_folder in sorted(os.listdir(increments_path)):
    if batch_folder.startswith("batch_"):
        year = batch_folder.replace("batch_", "")
        orders_path = os.path.join(increments_path, batch_folder, "orders.parquet")
        
        if os.path.exists(orders_path):
            inc_orders = spark.read.parquet(orders_path)
            inc_max_orderkey = inc_orders.agg(F.max("o_orderkey")).collect()[0][0]
            inc_max_orderdate = inc_orders.agg(F.max("o_orderdate")).collect()[0][0]
            
            print(f"\n📁 {batch_folder} (Year {year}):")
            print(f"   Max o_orderkey: {inc_max_orderkey}")
            print(f"   Max o_orderdate: {inc_max_orderdate}")

spark.stop()


BASE LAYER (Historical Data)
Max o_orderkey: 5999975
Max o_orderdate: 1995-12-31

INCREMENTAL BATCHES (By Year)

📁 batch_1996 (Year 1996):
   Max o_orderkey: 6000000
   Max o_orderdate: 1996-12-31

📁 batch_1997 (Year 1997):
   Max o_orderkey: 5999973
   Max o_orderdate: 1997-12-31

📁 batch_1998 (Year 1998):
   Max o_orderkey: 5999943
   Max o_orderdate: 1998-08-02
